# 04 オートエンコーダ — 正常データを「覚えて」、復元できない日を異常とする

## ゴール
- **オートエンコーダ**（深層学習）で異常を検知する仕組みを理解する。
- 「正常データだけで学習する」ことの意味と、**異常を覚えてしまう落とし穴**を、実験で確かめる。
- 02・03 の手法と比べて、**深層学習が役に立つ場面と、そうでない場面**を知る。

## 前のNotebookとのつながり
02 は「期待値モデル」、03 は「分布からの外れ」で検知しました。
ここでは、**正常データを圧縮して復元するネットワーク**を学習し、うまく復元できない日を異常とします。

## やること
1. オートエンコーダを学習し、学習曲線を見る
2. 復元誤差（異常スコア）の時系列を見る
3. 圧縮の細さ・入力の窓・学習データの汚れを変えて、10系列 × 3シードで評価する
4. 学習データに異常が混ざると何が起きるかを確かめる

## キーワード
- **オートエンコーダ**: 入力を、細い通路（コード）に**圧縮**してから、**元に戻す**ネットワーク。
- **復元誤差**: 入力と、復元した出力との差。大きいほど「学習した正常のパターンから外れている」。
- **ボトルネック（code）**: 圧縮した後の次元数。細いほど、正常の本質的なパターンだけを残す。
- **過学習・恒等写像**: 通路が太すぎると、何でもそのまま復元できてしまい、異常も復元してしまうこと。
- **汚染**: 学習データに異常が混ざること。現実の「正常データ」は、完全に正常とは限らない。

In [ ]:
import sys
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT / "src"))
from anomaly_synth import make_split
from anomaly_eval import alarm_by_normal_quantile
from baselines import DETECTOR_COLUMNS
from ae import DAY_FEATURES, AEDetector, day_features, make_windows, out_of_fold_score
import run_phase4

plt.rcParams["font.family"] = ["Hiragino Sans", "AppleGothic", "DejaVu Sans"]
train = make_split("train")[DETECTOR_COLUMNS]
val = make_split("val", 1)
x = val[DETECTOR_COLUMNS]

f = day_features(x)
print("1日の特徴量:", DAY_FEATURES, "→", f.shape)
print("7日窓にすると:", make_windows(f, 7).shape, "（1行 = 直近7日 × 8特徴 = 56個の数字）")

**何をしているか**: オートエンコーダへの入力を作ります。1日ぶんは8個の数字（発電量・日射量・日照率・天気・季節・発電量÷日射量）です。
7日窓にすると、直近7日ぶんを並べた56個の数字になります。

**出力の見方**: 1行が1日に対応し、366行あります。窓にすると列が増え、直近の変化も見られます。

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(11, 3.8))
for code_size in (2, 16):
    det = AEDetector(window=1, code=code_size, seed=0).fit(train)
    ax[0].plot(det.history, label=f"code={code_size}（学習の最終誤差 {det.final_loss:.3f}）")
ax[0].set(title="学習曲線", xlabel="エポック", ylabel="復元誤差（学習データ）", yscale="log"); ax[0].legend()

det = AEDetector(window=1, code=2, seed=0).fit(train)
thr = alarm_by_normal_quantile(out_of_fold_score(train, window=1, code=2, seed=0), 0.03)
score = det.score(x)
ax[1].plot(val["date"], score, color="gray", lw=0.8)
colors = {"outage": "red", "degrade": "orange", "shade": "purple", "stuck": "green"}
for kind, c in colors.items():
    d = val[val["anomaly_type"] == kind]
    ax[1].scatter(d["date"], score[d.index], s=12, color=c, label=kind, zorder=3)
ax[1].axhline(thr, color="k", ls="--", lw=1, label="しきい値")
ax[1].set(title="復元誤差（code=2）。大きいほど異常", yscale="log"); ax[1].legend(fontsize=7, ncol=2)
fig.tight_layout()

**何をしているか**: 左は、学習が進むにつれて復元誤差が下がる様子（学習曲線）です。通路が細い（code=2）場合と、太い（code=16）場合を比べます。
右は、val の1系列で、日ごとの復元誤差（異常スコア）を描いたものです。

**出力の見方**:
- 左: どちらも誤差が下がります。**太い通路（code=16）の方が、はるかに小さい誤差**まで下がります。入力の8個より広い通路なので、ほぼ丸ごと写せてしまうからです。
- 右: 停電（赤）・影（紫）・固着（緑）の日は、誤差が正常日より大きく外れます。劣化（橙）は、正常日と混ざって見えます。
- 注意: **学習時の誤差が小さい＝良い異常検知器、ではありません**。異常まで復元してしまうと、誤差が出ず、見逃します。

In [ ]:
res, agg, chance = run_phase4.run(verbose=False)
agg

**何をしているか**: 6つの設定のオートエンコーダを、10系列 × 3つのシード（初期値の乱数）で評価します。しきい値は 02・03 と同じ（学習に使っていない年の、復元誤差の上位3%）です。最下行は、比較のための 02 の残差法です。

**出力の見方**:
- 1日入力の3つ（code=2, 4, 16）の `auprc` は0.51〜0.62で、残差法（0.687）より低めです。
  **圧縮の細さによる差は、シード間のばらつき（`auprc_seed_sd` 0.03〜0.07）や系列間の標準偏差（0.06〜0.08）と同程度**で、細い方が良い、とは言えません。
- **シードを変えるだけで、結果が0.03〜0.07ぶれます**（1日入力の設定）。線形回帰の残差法は学習が決定的で、シードによるぶれがありません。**オートエンコーダは、何度か学習して見る必要があります**。
- 7日窓の2つは、`auprc` が0.29〜0.30と低く、誤警報が月4.5〜4.8回と多く見えます。`fa_grace7`（異常直後の7日を除いたもの）は0.6〜0.7回なので、これまでと同様に、**窓による余韻**が原因です。
- **汚染学習（学習データに異常を混ぜた）**は、`auprc` が0.383で、同じ設定の正常のみ学習（0.505）より悪化し、シード間のばらつきも最大（0.112）です。次で詳しく見ます。

In [ ]:
print("== 検知率（イベント単位） ==")
print(res.pivot(index="detector", columns="type", values="recall").loc[agg.index])
print("\n== 日単位の検知率 ==")
print(res.pivot(index="detector", columns="type", values="day_recall").loc[agg.index])
print("\n偶然の水準（警報が独立にランダムに出た場合）:", chance.to_dict())

**出力の見方**:
- **停電**: 汚染学習以外は、イベント単位0.96〜1.0・日単位0.96〜1.0で、ほぼ確実に見つかります。
- **影**: 1日入力のcode=2は、イベント単位0.98・日単位0.90、code=16は1.00・0.74、code=4は0.78・0.47です。**同じ手法でも、設定とシードで大きくぶれます**。
- **固着**: 汚染学習以外は、イベント単位0.95〜1.0・日単位0.66〜0.74です。残差法（日単位0.36）より良く、LOF（0.77〜0.82）よりはやや低めです。
- **劣化**: 日単位は0.01〜0.12で、**ほぼ見つかっていません**（残差法は0.29）。7日窓の劣化のイベント単位検知率（0.10〜0.40）が偶然の水準（0.64）を下回るのは、窓の警報が連続して（まとまって）出るため、偶然でも「1回は当たる」確率が計算より低くなるからだと考えられます。**この偶然の水準は、警報が独立に出る前提の目安**にすぎません。
- 全体として、**停電・影・固着は見つかり、劣化は見つかりません**。02・03 と同じ傾向です。

In [ ]:
rows = []
for name, window, code_size, data in [("正常のみ", 1, 4, "clean"), ("異常が混ざる", 1, 4, "dirty")]:
    tr = train if data == "clean" else run_phase4.dirty_train()
    ratios = {k: [] for k in ["outage", "shade", "stuck"]}
    normal_err = []
    for seed in (0, 1, 2):
        s = AEDetector(1, code_size, seed=seed).fit(tr).score(x)
        base = s[val["is_anomaly"] == 0].median()
        normal_err.append(base)
        for k in ratios:
            ratios[k].append(s[val["anomaly_type"] == k].median() / base)
    rows.append(dict(学習データ=name, 正常日の誤差の中央値=round(float(np.mean(normal_err)), 3),
                     **{f"{k}の誤差÷正常日の誤差": round(float(np.mean(v)), 1) for k, v in ratios.items()}))
pd.DataFrame(rows)

**何をしているか**: 同じ設定（1日入力・code=4）で、学習データが「正常のみ」の場合と「異常が約15%混ざった」場合を比べます。
各異常の復元誤差が、正常日の何倍になるかを、3シードの平均で見ます（倍率が大きいほど、見分けやすい）。

**出力の見方**:
- 正常日の復元誤差の中央値は、ほぼ同じ（0.053と0.050）です。一方、**異常日の倍率は大きく下がります**（停電125倍→19倍、影8.8倍→2.1倍、固着3.3倍→1.3倍）。異常を含むデータで学習すると、異常の日も復元できるようになり、誤差が正常日に近づくためです。
- つまり、オートエンコーダは**学習データの異常を「正常の一部」として覚えてしまいます**。
- 現実の「正常データ」は、ラベルがないので、**異常が混ざっていることが多い**でしょう。
  これが、教師なし手法の運用上の大きな課題です。学習データを、**目視や別の方法で確認してから使う**、または**外れ値を除いて学習し直す**といった対策が必要になります。

## 試してみよう（演習）
1. **学習のエポック数を変える**: `src/ae.py` の `epochs=300` を `30` と `1500` に変えて、`python src/run_phase4.py` を実行してください。
   見るポイントは、`auprc` と、停電・影・固着の日単位の検知率です。分かるのは、学習が足りないと正常も復元できず誤警報が増え、学習しすぎると異常まで復元することです。
2. **異常の混ざる割合を変える**: `src/run_phase4.py` の `dirty_train()` で、`make_split("val", 100)` を `make_split("val", 100, {"outage": 1})` のように変えて（異常を減らして）、汚染学習の結果が正常のみ学習に近づくか確かめてください。分かるのは、汚染の割合と検知性能の低下の関係です。
3. **シードを増やす**: `SEEDS = (0, 1, 2)` を `(0, 1, 2, 3, 4)` にして、`auprc_seed_sd` が変わるか見てください。分かるのは、ばらつきが偶然の範囲か、設定に由来するかです。

## まとめ
- オートエンコーダは、停電・影・固着を見つけられたが、**劣化は見つけられず**、02の残差法（日単位0.29）に及ばなかった。`auprc` は、残差法（0.687）が最も高く、LOF・特徴B（0.649）とオートエンコーダ（0.50〜0.62）はそれ以下か同程度だった。
- **シードによる結果のばらつきが大きく**（auprc 0.03〜0.07）、設定の差を判断しにくい。何度か学習して見る必要がある。
- **学習データに異常が混ざると、異常を覚えてしまい**、性能が大きく落ちる。「正常データだけで学習する」は、現実には簡単でない。
- この合成データのように、**期待値を簡単に設計できる**問題では、深層学習の出番は少ない。出番があるのは、期待値を設計しにくい複雑な問題（多数のセンサー、画像など）だろう。

次の Phase5 では、これまでの手法を**組み合わせて**、異常の種類ごとの検知率・誤警報・検知までの日数・劣化の検知限界を**まとめて評価**し、**最終評価（testは1回だけ）**を行います。